In [7]:
import pickle
import random
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from rdkit.Chem import AllChem
from rdchiral.main import rdchiralRunText

from anytree import Node, RenderTree, PreOrderIter
from tqdm import tqdm

/home/wang_m@WMGDS.WMG.WARWICK.AC.UK/anaconda3/envs/drug/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
fp_dim = 2048

# memo

In [9]:
class RolloutPolicyNet(nn.Module):
    def __init__(self, n_rules, fp_dim=2048, dim=512, dropout_rate=0.3):
        super(RolloutPolicyNet, self).__init__()
        self.fp_dim = fp_dim
        self.n_rules = n_rules
        self.dropout_rate = dropout_rate
        self.fc1 = nn.Linear(fp_dim,dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.dropout1 = nn.Dropout(dropout_rate)
        # self.fc2 = nn.Linear(dim,dim)
        # self.bn2 = nn.BatchNorm1d(dim)
        # self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(dim,n_rules)

    def forward(self,x, y=None, loss_fn =nn.CrossEntropyLoss()):
        x = self.dropout1(F.elu(self.bn1(self.fc1(x))))
        # x = self.dropout1(F.elu(self.fc1(x)))
        # x = self.dropout2(F.elu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        if y is not None :
            return loss_fn(x, y)
        else :
            return x
        return x

def load_model(state_path, template_rule_path,fp_dim=2048):
    template_rules = {}
    with open(template_rule_path, 'r') as f:
        for i, l in enumerate(f):
            rule= l.strip()
            template_rules[rule] = i
    idx2rule = {}
    for rule, idx in template_rules.items():
        idx2rule[idx] = rule
    rollout = RolloutPolicyNet(len(template_rules),fp_dim=fp_dim)
    
    state_dict = torch.load(state_path, map_location='cpu')
    # Remove the 'module.' prefix if present (i.e., if it was saved using DataParallel)
    new_state_dict = {}
    for key in state_dict.keys():
        new_key = key.replace("module.", "")  # Remove 'module.' prefix
        new_state_dict[new_key] = state_dict[key]
    rollout.load_state_dict(new_state_dict)
    return rollout, idx2rule

class MLPModel(object):
    def __init__(self, state_path, template_path, device=-1, fp_dim=2048, realistic_filter=False):
        super(MLPModel, self).__init__()
        self.fp_dim = fp_dim
        self.net, self.idx2rules = load_model(state_path, template_path, fp_dim)
        self.net.eval()
        self.device = device
        if device >= 0:
            self.net.to(device)

        self.realistic_filter = realistic_filter
        
        self.reference_net, _ = load_model(state_path, template_path, fp_dim)
        self.reference_net.eval()
        if device >= 0:
            self.reference_net.to(device)

    def forward_topk(self, arr, topk=10):
        preds = self.net(arr)
        preds_reference = self.reference_net(arr).detach()

        if self.realistic_filter:
            preds_reference, idx_topk = torch.topk(preds_reference, k=topk)
            preds = preds.gather(1, idx_topk)
        else:
            preds, idx_topk = torch.topk(preds, k=topk)
            preds_reference = preds_reference.gather(1, idx_topk)

        return preds, preds_reference, idx_topk

In [10]:
def preprocess(X,fp_dim):
    # Compute fingerprint from mol to feature
    mol = Chem.MolFromSmiles(X)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=int(fp_dim),useChirality=True)
    onbits = list(fp.GetOnBits())
    arr = np.zeros(fp.GetNumBits())
    arr[onbits] = 1
    # arr = (arr - arr.mean())/(arr.std() + 0.000001)
    # arr = arr / fp_dim
    # X = fps_to_arr(X)
    return arr

def prepare_starting_molecules(filename):
    if filename[-3:] == 'csv':
        starting_mols = set(list(pd.read_csv(filename)['mol']))
    else:
        assert filename[-3:] == 'pkl'
        with open(filename, 'rb') as f:
            starting_mols = pickle.load(f)
    return starting_mols

In [71]:
class RetroEnv:
    def __init__(self, mode='training'):
        self.mode = mode
        if mode == 'training':
            self.routes = pickle.load(open('dataset/routes_train.pkl', 'rb'))
            self.target_mols = [Chem.CanonSmiles(self.route[0].split('>')[0]) for self.route in tqdm(self.routes[:200], mininterval=2, desc='Load target molecules')]
        elif mode == 'evaluation':
            self.routes = pickle.load(open('dataset/routes_possible_test_hard.pkl', 'rb'))
            self.target_mols = [Chem.CanonSmiles(self.route[0].split('>')[0]) for self.route in tqdm(self.routes, mininterval=2, desc='Load target molecules')]
        self.building_blocks = prepare_starting_molecules('dataset/origin_dict_canonical.csv')
    
    def reset(self, mol_idx=None):
        self.tree_mdp = None
        self.tobevisited_nodes = []

        if mol_idx == None:
            self.obs = random.choice(self.target_mols)
        else:
            self.obs = self.target_mols[mol_idx]
            
        self.obs_node = Node(self.obs)
        self.tree_mdp = self.obs_node
        return self.obs

    def step(self, a):
        reactants = rdchiralRunText(a, self.obs)
        if len(reactants) == 0:
            return None, 0, 1, None
        else:
            reactants = list(set(reactants[0].split('.')))
        
        for r in reactants:
            r_node = Node(r, parent=self.obs_node)
            if not r in self.building_blocks:
                self.tobevisited_nodes.append(r_node)
        
        if self.tobevisited_nodes:
            self.obs = self.tobevisited_nodes[-1].name
            self.obs_node = self.tobevisited_nodes[-1]
            self.tobevisited_nodes.remove(self.tobevisited_nodes[-1])
            return self.obs, 0, 0, None
        else:
            self.obs = None
            self.obs_node = None
            return None, 1, 1, None

        #failed_leaves = [node for node in PreOrderIter(self.tree_mdp) if node.is_leaf and (node not in self.building_blocks)]
        
    


In [103]:
class RetroRL:
    def __init__(self, topk=10, fp_dim=2048, sampling=True):
        self.fp_dim = fp_dim
        self.topk = 10
        self.sampling = sampling
        self.one_step = MLPModel(state_path='one_step_model/saved_rollout_state_1_2048.ckpt', 
                                 template_path='one_step_model/template_rules_1.dat', device=0)
    def get_action(self, a): # a is a SMILES
        arr = preprocess(a, self.fp_dim)
        arr = np.reshape(arr,[-1, arr.shape[0]])
        arr = torch.tensor(arr, dtype=torch.float32)
        arr = arr.to(device='cuda')
        preds, preds_reference, idx = self.one_step.forward_topk(arr, topk=self.topk) 
        if self.sampling:
            probs = F.softmax(preds, dim=1)
            probs_reference = F.softmax(preds_reference, dim=1)
            action_class = torch.multinomial(probs, num_samples=1).squeeze().cpu().numpy()
            reference_class = torch.multinomial(probs_reference, num_samples=1).squeeze().cpu().numpy()
        else: 
            action_class = torch.argmax(preds, dim=1).cpu().numpy()
            reference_class = torch.argmax(preds_reference, dim=1).cpu().numpy()
        action_indices = idx[0][action_class].cpu().item()
        reference_indices = idx[0][reference_class].cpu().item()

        ac_reaction = self.one_step.idx2rules[action_indices]
        ref_reaction = self.one_step.idx2rules[reference_indices]
        return ac_reaction, ref_reaction

In [104]:
agent = RetroRL(sampling=False)
env = RetroEnv()
eval_env = RetroEnv(mode='evaluation')

Load target molecules: 100%|████████████████| 190/190 [00:00<00:00, 5273.76it/s]


In [105]:
def eval(envs, agent):
    eval_info = {}
    for env in envs:
        rewards = np.zeros((len(env.target_mols), 1))
        for mol_idx in tqdm(range(min(len(env.target_mols), 200)), desc=f'Evaluate the model ({env.mode})'):
            obs = env.reset(mol_idx)
            for t in range(100):
                a, _ = agent.get_action(obs)
                next_obs, reward, done, info = env.step(a)
                rewards[mol_idx] = reward
                if done:
                    break
                obs = next_obs
        eval_info[env.mode+'/success rate'] = rewards.mean()
    return eval_info

In [106]:
eval_info = eval([eval_env, env], agent)
print(eval_info)

Evaluate the model (training): 100%|██████████| 200/200 [00:06<00:00, 30.88it/s]

{'evaluation/success rate': 0.1631578947368421, 'training/success rate': 0.925}
